# Project 2 — Streaming Text Analytics and Performance Lab

Build a deterministic text-processing pipeline using generators, token normalization, frequency tables, caching policy, benchmarks, and memory profiling. The project emphasizes evidence-based performance decisions.

**Level:** Advanced portfolio · **Suggested time:** 4–8 hours · **Requirements:** Python 3.11+, Jupyter.

## Deliverables

- Transform large text through a lazy pipeline
- Use Counters and stable ranking for word frequencies
- Compare materialized and streaming alternatives
- Measure selected workloads with timeit and tracemalloc
- Write reproducible functional tests

**Operating rules:** All demonstrations are offline, deterministic, and side-effect-limited. Read the full requirements and design before changing code. For a portfolio submission, move reusable logic to Python modules and add tests and a short architecture decision record.

The runnable implementation is a worked example. Submit your own changes and regression tests in the independent section.

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "projects"))
    sys.path.insert(0, os.getcwd())

## Step 1 — Define a normalization contract

Decide whether punctuation, case, and Unicode normalization matter before coding. This example extracts alphanumeric Unicode words, applies casefold, and keeps each token's original occurrence in a lazy stream.

In [ ]:
import re
import unicodedata

def normalize_tokens(lines):
    for line in lines:
        normalized = unicodedata.normalize("NFKC", line).casefold()
        for token in re.findall(r"\w+", normalized, flags=re.UNICODE):
            if token:
                yield token

sample_lines = ["Python, PYTHON! and data.", "Data pipelines are useful."]
tokens = list(normalize_tokens(sample_lines))
print(tokens)
assert tokens.count("python") == 2

**Review question:** What linguistic complexities can simple regex tokenization mishandle?

## Step 2 — Streaming aggregation and deterministic ranking

Counter consumes an iterable and accumulates counts, while stable sorting gives reproducible results for equal frequencies. Choose a tie rule explicitly. Holding only counts can be smaller than storing every token for long input streams.

In [ ]:
from collections import Counter

def word_frequencies(lines):
    return Counter(normalize_tokens(lines))

def top_words(counts, limit=5):
    if limit < 0:
        raise ValueError("limit cannot be negative")
    return sorted(counts.items(), key=lambda item: (-item[1], item[0]))[:limit]

frequencies = word_frequencies(sample_lines)
print(top_words(frequencies))
assert frequencies["data"] == 2

**Review question:** When would the number of unique words approach the number of total tokens?

## Step 3 — Generator input and resource ownership

An iterable of lines can be a list, generator, or open file. Process it in one pass and avoid assuming len() is supported. Resource ownership remains with the caller; a file producer should close its file using with.

In [ ]:
def simulated_file_lines():
    for line in ("alpha beta", "beta gamma", "alpha"):
        yield line

result = word_frequencies(simulated_file_lines())
print(top_words(result))
assert result["alpha"] == 2

**Review question:** What could go wrong if a generator source is consumed twice?

## Step 4 — Benchmark fairly with timeit

Compare equivalent results on a representative dataset. timeit reports measurements for the current machine and interpreter—not universal ratios. Include memory costs and differences in readability when evaluating results.

In [ ]:
import timeit

text_lines = ["Python data " * 20 for _ in range(100)]
def list_pipeline():
    return Counter(list(normalize_tokens(text_lines)))
def generator_pipeline():
    return Counter(normalize_tokens(text_lines))

assert list_pipeline() == generator_pipeline()
list_s = timeit.timeit(list_pipeline, number=5)
stream_s = timeit.timeit(generator_pipeline, number=5)
print("materialized:", round(list_s, 5), "streamed:", round(stream_s, 5))

**Review question:** Why might the streaming method use less peak memory without always being faster?

## Step 5 — Memory observation with tracemalloc

tracemalloc measures Python-tracked allocations, not the complete memory of the process. To compare approaches fairly, start tracing before each workload and stop after capturing a peak. Avoid asserting that one approach always wins across environments.

In [ ]:
import tracemalloc

def trace_peak(callable_):
    tracemalloc.start()
    try:
        value = callable_()
        current, peak = tracemalloc.get_traced_memory()
        return value, peak
    finally:
        tracemalloc.stop()

counts, peak = trace_peak(generator_pipeline)
print("unique words:", len(counts), "traced peak bytes:", peak)
assert peak >= 0

**Review question:** Why can memory peaks vary even on an identical input?

## Step 6 — Property-style tests

Testing key invariants across several generated examples is more reliable than checking only a single sentence. The original count should equal the total number of emitted tokens, and casefolded variants should agree.

In [ ]:
import random
rng = random.Random(7)
for _ in range(30):
    size = rng.randint(0, 20)
    words = [rng.choice(["Ada", "ADA", "Python", "DATA"]) for _ in range(size)]
    actual = word_frequencies([" ".join(words)])
    assert sum(actual.values()) == size
    assert actual["ada"] == sum(word.casefold() == "ada" for word in words)
print("30 generated cases passed")

**Review question:** How would you test Unicode normalization corner cases systematically?

## Independent extensions

Choose at least two and implement with tests. Describe error behavior and any scaling limits.

**Review evidence.** For stop-word filtering, show that a stop word can be excluded on one run and counted on the next. Compare the results on the same text supplied as a list and as a one-pass iterator. Explain the decision behind your boundary case.

**Extension 1.** Implement an incremental top-k algorithm using heapq for a large vocabulary.

In [ ]:
# TODO: Extension 1.
# Add a reproducible normal and failure case.

**Extension 2.** Add configurable stop-word filtering without hidden mutable defaults.

In [ ]:
# TODO: Extension 2.
# Add a reproducible normal and failure case.

**Extension 3.** Measure memory for two different data-size regimes and discuss observed behavior.

In [ ]:
# TODO: Extension 3.
# Add a reproducible normal and failure case.

**Extension 4.** Create a CLI wrapping the text pipeline and writing a CSV of word frequencies.

In [ ]:
# TODO: Extension 4.
# Add a reproducible normal and failure case.

## Capstone rubric

- [ ] Requirements and public contracts documented.
- [ ] No leaked resources, orphaned tasks, or unbounded loops.
- [ ] Malformed inputs and failure paths covered by tests.
- [ ] No live network dependency or exposed secrets in test code.
- [ ] Rerunnable notebook without hidden state.
- [ ] Clear architecture explanation and trade-off analysis.

**Final:** Complete the Advanced assessment and prepare a project README highlighting your test evidence.